# Verification of the scenario data files

These data files include RMa, UMi, UMa files.

**Refactor notes (vs. the previous version of this notebook):**
- Each `.h5` file is opened and loaded into memory **once**, then cached in a
  `SCENARIOS` dict. Every later cell reads from that cache instead of
  re-opening the file from disk, which is what most of the earlier cells did.
- The "Label Integrity" cell had a bug: it reused a leftover `f`/`file`
  variable from an earlier loop instead of looping over all three files, so
  it silently only ever checked the last file. Fixed here.
- The PAPR check existed as three near-identical copy-pasted cells. Merged
  into one.
- The constellation plots now follow your teacher's approach: one panel per
  (modulation, SNR) pair, each showing a small, fixed number of *whole*
  waveform vectors rather than stacking many independent vectors from
  different channel draws into one overlapping scatter cloud.


## 1. Imports and Configuration

In [ ]:
import numpy as np
import h5py
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
DATA_DIR = '../data/'

SCENARIO_FILES = {
    'Rma': DATA_DIR + 'Rma.h5',
    'Uma': DATA_DIR + 'Uma.h5',
    'Umi': DATA_DIR + 'Umi.h5',
}

MOD_NAMES = ['BPSK', 'QPSK', '16QAM', '64QAM', '256QAM', '1024QAM']
EXPECTED_SNRS = np.arange(-10, 21, 2)   # -10, -8, ..., 20 -> 16 values
FOCUS_SNRS = [-10, 0, 10, 20]           # subset used for the constellation grids
TARGET_SNR = 20.0                       # SNR used for energy / PAPR checks

## 2. Load each scenario once

`load_scenario` reads the whole file a single time and pre-computes the
argmax modulation label per row, so nothing downstream needs to touch the
`.h5` file again.

In [ ]:
def load_scenario(path):
    """Load one scenario file fully into memory and pre-compute mod labels."""
    with h5py.File(path, 'r') as f:
        data = f['Data'][:]
        mods = f['Mods'][:]
        snrs = f['SNRs'][:]

    mod_idx = np.argmax(mods, axis=1)
    mod_labels = np.array(MOD_NAMES)[mod_idx]

    return {
        'path': path,
        'data': data,
        'mods': mods,
        'snrs': snrs,
        'mod_idx': mod_idx,
        'mod_labels': mod_labels,
    }

SCENARIOS = {}
for label, path in SCENARIO_FILES.items():
    size_mb = Path(path).stat().st_size / 1024**2
    print(f"{label:4s} {path}  ({size_mb:.2f} MB)")
    SCENARIOS[label] = load_scenario(path)

## 3. Initial Global Sanity Checks

In [ ]:
for label, s in SCENARIOS.items():
    print(f"---- {label} ----")
    print(f"Data shape/dtype: {s['data'].shape}, {s['data'].dtype}")
    print(f"Mods shape/dtype: {s['mods'].shape}, {s['mods'].dtype}")
    print(f"SNRs shape: {s['snrs'].shape}")
    row_sums = s['mods'].sum(axis=1)
    print(f"Mods row-sum range: [{row_sums.min()}, {row_sums.max()}]")
    print()

## 4. Checking for NaNs and Infs

In [ ]:
for label, s in SCENARIOS.items():
    print(f"---- {label} ----")
    for name in ('data', 'mods', 'snrs'):
        arr = s[name]
        print(f"{name:5s} - NaNs: {np.isnan(arr).any()}, Infs: {np.isinf(arr).any()}")
    print()

## 5. Label Integrity & Balancing

(Fixed to actually loop over every scenario — the original cell checked only
the last file loaded by a previous loop.)

In [ ]:
for label, s in SCENARIOS.items():
    mods = s['mods']
    row_sums = mods.sum(axis=1)

    n_total = len(row_sums)
    n_exact = np.sum(row_sums == 1.0)
    is_one_hot = np.all(np.isin(mods, [0.0, 1.0])) and np.all(row_sums == 1.0)

    print(f"---- {label} ----")
    print(f"Rows: {n_total}")
    print(f"  Sum exactly 1.0: {n_exact}/{n_total}")
    print(f"  Strict one-hot (only 0/1 values, single 1 per row): {is_one_hot}")

    if n_exact != n_total:
        bad_idx = np.where(row_sums != 1.0)[0]
        print(f"  WARNING: {len(bad_idx)} bad rows, e.g. indices {bad_idx[:10]}")
        print(f"  Example bad row sums: {row_sums[bad_idx[:5]]}")
    print()

## 6. SNR Grid Check

In [ ]:
for label, s in SCENARIOS.items():
    snrs = s['snrs']
    unique_snrs, counts = np.unique(snrs, return_counts=True)

    print(f"---- {label} ----")
    print(f"Unique SNR count: {len(unique_snrs)} (expected {len(EXPECTED_SNRS)})")
    print(f"Matches expected grid -10..20 step 2: {np.array_equal(unique_snrs, EXPECTED_SNRS)}")
    all_equal = len(set(counts)) == 1
    print(f"All per-SNR counts equal: {all_equal}" + (f" (each = {counts[0]})" if all_equal else f" ({dict(zip(unique_snrs, counts))})"))
    print()

## 7. Cross-Tabulation Matrix (SNR vs Modulation)

In [ ]:
for label, s in SCENARIOS.items():
    df = pd.DataFrame({'SNR': s['snrs'], 'Mod': s['mod_labels']})
    crosstab = pd.crosstab(df['SNR'], df['Mod'])[MOD_NAMES]

    print(f"---- {label} ----")
    print(crosstab)

    expected_count = crosstab.values[0, 0]
    all_equal = (crosstab.values == expected_count).all()
    print(f"\nAll cells == {expected_count}: {all_equal}")
    if not all_equal:
        for r, c in np.argwhere(crosstab.values != expected_count):
            print(f"  MISMATCH: SNR={crosstab.index[r]}, Mod={crosstab.columns[c]}, count={crosstab.values[r, c]}")

    plt.figure(figsize=(8, 6))
    sns.heatmap(crosstab, annot=True, fmt='d', cmap='viridis', cbar_kws={'label': 'Sample Count'})
    plt.title(f"SNR x Modulation Cross-Tabulation — {label}")
    plt.xlabel("Modulation")
    plt.ylabel("SNR (dB)")
    plt.tight_layout()
    plt.show()
    print()

## 8. Cross-Modulation Energy Consistency @ 20 dB

In [ ]:
for label, s in SCENARIOS.items():
    print(f"---- {label} @ {TARGET_SNR} dB ----")
    mask_snr = np.isclose(s['snrs'], TARGET_SNR)

    energies = {}
    for m in MOD_NAMES:
        mask = mask_snr & (s['mod_labels'] == m)
        sig = s['data'][mask]
        energies[m] = np.mean(np.abs(sig) ** 2)
        print(f"  {m:8s}: mean energy = {energies[m]:.4f}  (n={mask.sum()})")

    vals = np.array(list(energies.values()))
    print(f"  --> Spread: min={vals.min():.4f}, max={vals.max():.4f}, ratio max/min={vals.max()/vals.min():.3f}\n")

## 9. Peak-to-Average Power Ratio (PAPR)

(The original notebook ran this same check three times in a row with
identical code — merged into one cell.)

In [ ]:
def compute_papr_db(signal):
    """Mean PAPR in dB for a batch of complex vectors, shape (N, T)."""
    power = np.abs(signal) ** 2
    papr_linear = power.max(axis=1) / power.mean(axis=1)
    return 10 * np.log10(papr_linear.mean())

QAM_ORDER = ['16QAM', '64QAM', '256QAM', '1024QAM']

for label, s in SCENARIOS.items():
    mask_snr = np.isclose(s['snrs'], TARGET_SNR)

    papr_db = {
        m: compute_papr_db(s['data'][mask_snr & (s['mod_labels'] == m)])
        for m in MOD_NAMES
    }

    print(f"---- {label} @ {TARGET_SNR} dB ----")
    print("PAPR (dB): " + ", ".join(f"{m}={papr_db[m]:.2f}" for m in MOD_NAMES))

    gap = papr_db['BPSK'] - papr_db['QPSK']
    print(f"  BPSK - QPSK = {gap:+.2f} dB ({'BPSK higher' if gap > 0 else 'QPSK higher or equal'})")

    qam_vals = [papr_db[m] for m in QAM_ORDER]
    monotonic = all(qam_vals[i] < qam_vals[i + 1] for i in range(len(qam_vals) - 1))
    print(f"  QAM monotonic increasing: {monotonic}\n")

## 10. Constellation Grids — one clean vector per (Modulation, SNR)

This is the approach your teacher suggested: instead of stacking many
*independent* vectors (each its own noise/fading draw) into one overlapping
scatter plot, each panel below shows `n_vectors` full waveform(s) for a
single modulation/SNR combination. That keeps each panel legible and makes
it easy to compare shape and spread across modulations (rows) and SNRs
(columns) directly.

In [ ]:
def plot_constellation_grid(label, s, snrs=FOCUS_SNRS, mods=MOD_NAMES, axis_limit=3, n_vectors=1):
    """Grid of constellation scatter plots: rows = modulations, cols = SNRs.

    Each panel plots `n_vectors` full data vectors for that exact
    (modulation, SNR) pair — never a mix of vectors from different SNRs or
    modulations, and never so many independent vectors that the plot turns
    into an uninterpretable blob.
    """
    n_mods, n_snrs = len(mods), len(snrs)
    fig, axes = plt.subplots(n_mods, n_snrs, figsize=(3.2 * n_snrs, 3.2 * n_mods), squeeze=False)
    fig.suptitle(f"{label} — constellations across SNR", y=1.01)

    for i, mod_name in enumerate(mods):
        mod_mask = s['mod_labels'] == mod_name

        for j, snr in enumerate(snrs):
            ax = axes[i, j]
            row_mask = mod_mask & np.isclose(s['snrs'], snr)
            available = s['data'][row_mask]

            if len(available) == 0:
                ax.set_visible(False)
                continue

            sample = available[:n_vectors].flatten()
            ax.scatter(sample.real, sample.imag, s=3, alpha=0.4)
            ax.axhline(0, color='gray', lw=0.5)
            ax.axvline(0, color='gray', lw=0.5)
            ax.set_xlim(-axis_limit, axis_limit)
            ax.set_ylim(-axis_limit, axis_limit)
            ax.set_aspect('equal')
            ax.grid(alpha=0.2)

            if i == 0:
                ax.set_title(f"{snr:g} dB", fontsize=10)
            if j == 0:
                ax.set_ylabel(mod_name, fontsize=10)
            if i == n_mods - 1:
                ax.set_xlabel("I")

    plt.tight_layout()
    plt.show()

for label, s in SCENARIOS.items():
    plot_constellation_grid(label, s)

## 11. Fading Severity: Amplitude vs Time for a single vector

In [ ]:
def plot_amplitude_trace(mod='64QAM', snr=20.0, vector_idx=0):
    fig, axes = plt.subplots(len(SCENARIOS), 1, figsize=(14, 3.2 * len(SCENARIOS)), sharex=True, sharey=True)

    for ax, (label, s) in zip(axes, SCENARIOS.items()):
        mask = np.isclose(s['snrs'], snr) & (s['mod_labels'] == mod)
        vec = s['data'][mask][vector_idx]
        amplitude = np.abs(vec)
        cv = amplitude.std() / amplitude.mean()

        ax.plot(amplitude, lw=1)
        ax.set_title(f"{label} — |signal| over time — {mod} @ {snr} dB")
        ax.set_ylabel("Amplitude")
        ax.grid(alpha=0.3)
        ax.text(0.98, 0.9, f"CV={cv:.3f}", transform=ax.transAxes, ha='right', va='top',
                fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

    axes[-1].set_xlabel("Time step")
    plt.tight_layout()
    plt.show()

plot_amplitude_trace()

## 12. Fading Volatility, Quantified

`fading_cv` returns the per-vector coefficient of variation of `|amplitude|`
within each vector — one flexible function replaces what used to be three
separate cells (single-SNR summary, single-SNR-with-detail, and a
multi-SNR loop calling the same logic again).

In [ ]:
def fading_cv(s, mod, snr, n_vectors=200):
    """Per-vector coefficient of variation of |amplitude| within each vector."""
    mask = np.isclose(s['snrs'], snr) & (s['mod_labels'] == mod)
    vecs = s['data'][mask][:n_vectors]
    amps = np.abs(vecs)
    return np.std(amps, axis=1) / np.mean(amps, axis=1)

for snr in [20.0, 10.0, 0.0]:
    print(f"=== 64QAM @ {snr} dB ===")
    for label, s in SCENARIOS.items():
        cvs = fading_cv(s, mod='64QAM', snr=snr)
        print(f"  {label:4s}: mean CV = {cvs.mean():.4f}, std across vectors = {cvs.std():.4f}")
    print()

## 13. Summary: Fading Volatility Across Environments and SNRs

In [ ]:
rows = []
for label, s in SCENARIOS.items():
    for snr in EXPECTED_SNRS:
        cvs = fading_cv(s, mod='64QAM', snr=snr)
        rows.append({'File': label, 'SNR': snr, 'MeanCV': cvs.mean()})

summary_df = pd.DataFrame(rows)
pivot_table = summary_df.pivot(index='SNR', columns='File', values='MeanCV')

print("Mean fading CV by scenario and SNR — 64QAM\n")
print(pivot_table.round(4))

plt.figure(figsize=(7, 8))
sns.heatmap(pivot_table, annot=True, fmt='.3f', cmap='rocket_r',
            cbar_kws={'label': 'Mean CV (higher = more volatile fading)'})
plt.title("Fading Volatility Across Environments — 64QAM")
plt.xlabel("Scenario")
plt.ylabel("SNR (dB)")
plt.tight_layout()
plt.show()